# 🔴 Solution: Implement RMSNorm（参考版）

## 📋 题目描述

**难度：** Medium

实现 RMSNorm（均方根层归一化）。

RMSNorm 是 LayerNorm 的简化替代，跳过均值减法，仅通过激活值的均方根进行归一化。

**签名:** `rms_norm(x, weight, eps=1e-6) -> Tensor`

**参数:**
- `x` — 输入张量 (..., D)
- `weight` — 可学习的缩放参数 (D,)
- `eps` — 数值稳定性的 epsilon

**返回:** 归一化后的张量，形状与 x 相同

**约束:**
- `RMS(x) = sqrt(mean(x^2) + eps)` 沿最后一维
- 输出：`x / RMS(x) * weight`

**提示：** `rms = sqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)`
`return x / rms * weight`  ← 无需减均值


In [ ]:
import torch
import math

In [ ]:
# ✅ SOLUTION

def rmsnorm(x: torch.Tensor, weight: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    # RMS Norm（Root Mean Square Normalization）
    # 比 LayerNorm 更简单：不减均值，只除以 RMS
    # 公式：y = x / RMS(x) * γ，其中 RMS(x) = √(mean(x²))
    # 优点：计算更快（省去均值计算），在 LLM 中广泛使用（LLaMA, GPT-NeoX 等）

    # ---- Step 1: 计算均方值 ----
    # x.pow(2) 或 x ** 2：逐元素平方
    # .mean(dim=-1, keepdim=True)：沿最后一维求均值
    # 例如 x shape=[32, 768] → rms_sq shape=[32, 1]
    rms_sq = x.pow(2).mean(dim=-1, keepdim=True)

    # ---- Step 2: 计算 RMS 并归一化 ----
    # rms = √(mean(x²))，加 eps 防止除零
    # x / rms：将每个向量缩放到 RMS=1
    x_norm = x / torch.sqrt(rms_sq + eps)

    # ---- Step 3: 仿射变换 ----
    # 只有缩放 γ，没有偏移 β（比 LayerNorm 少一个参数）
    # 这是 RMSNorm 的设计选择，减少参数量
    return weight * x_norm

In [ ]:
x = torch.randn(2, 8)
out = rms_norm(x, torch.ones(8))
print('RMS of output:', out.pow(2).mean(dim=-1).sqrt())

## 📝 核心思路总结

1. **RMS 公式**：√(mean(x²))，不减均值，比 LayerNorm 更简洁
2. **只有 γ 没有 β**：减少参数，计算更快
3. **数值稳定**：eps 防止除零
4. **现代 LLM 标配**：LLaMA、GPT-NeoX 等都用 RMSNorm